In [ ]:
import ee
import time

In [ ]:
# initialize Earth Engine using a registered Google Cloud project that has
# the Earth Engine API enabled
ee.Initialize(project='your-project-name') 

# define region of interest 
roi = ee.FeatureCollection('TIGER/2018/States') \
        .filter(ee.Filter.eq('NAME', 'Maryland')) \
        .geometry()

# import MODIS LST dataset
modis = ee.ImageCollection('MODIS/061/MOD11A2')

month_names = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

In [ ]:
def process_and_export(year, month):
    """
    This function filters the daytime MODIS Land Surface Temperature (LST) 
    dataset for the specified year and month, computes the monthly mean 
    daytime LST in degrees Celsius, clips the mean image to the Maryland 
    boundary, and exports the result to Google Drive as a GeoTIFF image.

    Parameters
    ----------
    year : int
        The four-digit year (e.g., 2018).
    month : int
        The numeric month (1–12).

    Returns
    -------
    ee.batch.Task
        A started Earth Engine export task for the specified month.
    """
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    
    images = modis.filterDate(start_date, end_date).select('LST_Day_1km')
    images_mean = images.map(lambda img: img.multiply(0.02).subtract(273.15)) \
                        .mean() \
                        .clip(roi) \
                        .set('system:time_start', start_date.millis()) \
                        .set('system:time_end', end_date.millis())

    month_name = month_names[month - 1]
    description = f"{month_name}_{year}_Day_Mean_LST"

    task = ee.batch.Export.image.toDrive(
        image=images_mean,
        description=description,
        folder='Day_monthlymean', 
        region=roi,
        scale=1000,
        crs='EPSG:4326'
    )

    task.start()
    print(f"Started export: {description}")
    return task

In [ ]:
tasks = []

# create export tasks
for year in range(2018, 2021):
    for month in range(1, 13):
        if year == 2020 and month > 1:
            break
        task = process_and_export(year, month)
        tasks.append(task)

# monitor tasks..
while any(task.status()['state'] in ['READY', 'RUNNING'] for task in tasks):
    print('Waiting for tasks to complete...')
    time.sleep(60)  

print('All tasks complete.')